# AI 모델 검증 및 평가 예제 - PyTorch 버전

이 노트북은 AI 모델 검증과 평가의 기본 흐름을 PyTorch로 실습하는 예제입니다.

실습 내용은 다음과 같습니다.

1. 데이터 준비
2. 학습 데이터, 검증 데이터, 테스트 데이터 분리
3. PyTorch 모델 정의
4. 학습 중 검증 손실과 검증 정확도 확인
5. 테스트 데이터 최종 평가
6. Accuracy, Precision, Recall, F1-score, Confusion Matrix, ROC-AUC 계산
7. 학습 곡선 시각화

분류 문제 예제로 `Breast Cancer` 데이터셋을 사용합니다.

In [ ]:
# ============================================================
# 1. 필요한 라이브러리 불러오기
# ============================================================

# numpy는 배열 계산을 위한 기본 라이브러리입니다.
import numpy as np

# pandas는 표 형태의 데이터를 다룰 때 사용합니다.
import pandas as pd

# matplotlib은 그래프를 그릴 때 사용합니다.
import matplotlib.pyplot as plt

# PyTorch의 핵심 라이브러리입니다.
import torch

# torch.nn은 신경망 계층, 손실함수 등을 만들 때 사용합니다.
import torch.nn as nn

# torch.optim은 모델의 가중치를 업데이트하는 최적화 알고리즘을 제공합니다.
import torch.optim as optim

# TensorDataset은 입력 데이터와 정답 데이터를 하나의 데이터셋으로 묶을 때 사용합니다.
from torch.utils.data import TensorDataset

# DataLoader는 데이터를 미니배치 단위로 나누어 모델에 공급합니다.
from torch.utils.data import DataLoader

# sklearn의 유방암 데이터셋을 불러옵니다.
from sklearn.datasets import load_breast_cancer

# train_test_split은 데이터를 학습/검증/테스트 데이터로 나눌 때 사용합니다.
from sklearn.model_selection import train_test_split

# StandardScaler는 입력 특성값을 평균 0, 표준편차 1로 표준화합니다.
from sklearn.preprocessing import StandardScaler

# 평가 지표를 계산하기 위한 함수들입니다.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

# 실행할 때마다 최대한 같은 결과가 나오도록 난수를 고정합니다.
np.random.seed(42)

# PyTorch의 난수를 고정합니다.
torch.manual_seed(42)

# GPU가 사용 가능하면 GPU를 사용하고, 아니면 CPU를 사용합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 사용 장치를 출력합니다.
print("사용 장치:", device)

In [ ]:
# ============================================================
# 2. 데이터 불러오기
# ============================================================

# Breast Cancer 데이터셋을 불러옵니다.
# 이 데이터셋은 종양이 악성인지 양성인지 분류하는 이진 분류 데이터입니다.
data = load_breast_cancer()

# 입력 데이터 X를 가져옵니다.
# X는 종양의 반지름, 질감, 면적 등 여러 개의 수치 특성으로 구성됩니다.
X = data.data

# 정답 데이터 y를 가져옵니다.
# y는 0 또는 1로 구성된 클래스 라벨입니다.
y = data.target

# 특성 이름을 확인하기 위해 DataFrame으로 변환합니다.
df = pd.DataFrame(X, columns=data.feature_names)

# 정답 컬럼을 추가합니다.
df["target"] = y

# 데이터의 앞부분 5개 행을 출력합니다.
df.head()

In [ ]:
# ============================================================
# 3. 학습 데이터, 검증 데이터, 테스트 데이터 분리
# ============================================================

# 먼저 전체 데이터를 학습+검증 데이터와 테스트 데이터로 나눕니다.
# test_size=0.15는 전체 데이터의 15%를 최종 테스트 데이터로 사용한다는 의미입니다.
# stratify=y는 클래스 비율이 나누어진 데이터에서도 최대한 유지되도록 합니다.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# 학습+검증 데이터를 다시 학습 데이터와 검증 데이터로 나눕니다.
# 전체 기준으로 약 70% 학습, 15% 검증, 15% 테스트가 되도록 나눕니다.
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.1765,
    random_state=42,
    stratify=y_train_val
)

# 각 데이터 크기를 출력합니다.
print("학습 데이터 크기:", X_train.shape)
print("검증 데이터 크기:", X_val.shape)
print("테스트 데이터 크기:", X_test.shape)

# 클래스 분포를 확인합니다.
print("학습 정답 분포:", np.bincount(y_train))
print("검증 정답 분포:", np.bincount(y_val))
print("테스트 정답 분포:", np.bincount(y_test))

In [ ]:
# ============================================================
# 4. 입력 데이터 표준화
# ============================================================

# StandardScaler 객체를 생성합니다.
scaler = StandardScaler()

# 학습 데이터의 평균과 표준편차를 계산하고, 학습 데이터를 표준화합니다.
# 검증/테스트 데이터 정보가 학습 과정에 섞이면 안 되므로 fit은 학습 데이터에만 적용합니다.
X_train_scaled = scaler.fit_transform(X_train)

# 검증 데이터는 학습 데이터에서 계산한 평균과 표준편차를 사용하여 변환만 합니다.
X_val_scaled = scaler.transform(X_val)

# 테스트 데이터도 학습 데이터 기준으로 변환만 합니다.
X_test_scaled = scaler.transform(X_test)

In [ ]:
# ============================================================
# 5. NumPy 데이터를 PyTorch Tensor로 변환
# ============================================================

# 입력 데이터는 실수형이므로 float32 타입의 Tensor로 변환합니다.
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)

# 정답 데이터는 0 또는 1이지만, BCEWithLogitsLoss를 사용할 것이므로 float32로 변환합니다.
# view(-1, 1)은 정답 모양을 [샘플 수, 1] 형태로 맞춥니다.
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

# 검증 입력 데이터를 Tensor로 변환합니다.
X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)

# 검증 정답 데이터를 Tensor로 변환합니다.
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

# 테스트 입력 데이터를 Tensor로 변환합니다.
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

# 테스트 정답 데이터를 Tensor로 변환합니다.
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# TensorDataset은 입력과 정답을 하나로 묶어줍니다.
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

# DataLoader는 데이터를 batch_size 개수만큼 묶어서 모델에 넣어줍니다.
# shuffle=True는 매 epoch마다 데이터 순서를 섞어 학습 안정성을 높입니다.
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [ ]:
# ============================================================
# 6. PyTorch 분류 모델 정의
# ============================================================

# nn.Module을 상속받아 사용자 정의 신경망 클래스를 만듭니다.
class BinaryClassifier(nn.Module):
    # __init__은 모델 구조를 정의하는 부분입니다.
    def __init__(self, input_dim):
        # 부모 클래스 nn.Module의 초기화 함수를 실행합니다.
        super(BinaryClassifier, self).__init__()

        # Sequential은 여러 계층을 순서대로 연결하는 컨테이너입니다.
        self.model = nn.Sequential(
            # 첫 번째 완전연결층입니다.
            # input_dim개의 입력 특성을 받아 32개의 값을 출력합니다.
            nn.Linear(input_dim, 32),

            # ReLU는 음수는 0으로 만들고 양수는 그대로 통과시키는 활성화 함수입니다.
            nn.ReLU(),

            # 과적합을 줄이기 위해 일부 뉴런을 무작위로 꺼주는 Dropout입니다.
            nn.Dropout(0.2),

            # 두 번째 완전연결층입니다.
            nn.Linear(32, 16),

            # 두 번째 활성화 함수입니다.
            nn.ReLU(),

            # 출력층입니다.
            # 이진 분류이므로 출력값은 1개입니다.
            # 여기서는 Sigmoid를 붙이지 않습니다.
            # BCEWithLogitsLoss가 내부적으로 Sigmoid 계산을 포함하기 때문입니다.
            nn.Linear(16, 1)
        )

    # forward는 입력 데이터가 모델을 통과하는 흐름을 정의합니다.
    def forward(self, x):
        # 위에서 정의한 Sequential 모델에 입력 x를 넣고 결과를 반환합니다.
        return self.model(x)

# 입력 특성 개수를 구합니다.
input_dim = X_train_tensor.shape[1]

# 모델 객체를 생성하고 device로 이동합니다.
model = BinaryClassifier(input_dim).to(device)

# 모델 구조를 출력합니다.
print(model)

In [ ]:
# ============================================================
# 7. 손실함수와 최적화 알고리즘 정의
# ============================================================

# BCEWithLogitsLoss는 이진 분류에서 사용하는 손실함수입니다.
# 출력층의 raw score(logit)에 Sigmoid를 내부적으로 적용한 뒤 Binary Cross Entropy를 계산합니다.
criterion = nn.BCEWithLogitsLoss()

# Adam은 많이 사용하는 최적화 알고리즘입니다.
# lr은 learning rate, 즉 가중치를 한 번에 얼마나 수정할지 정하는 값입니다.
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# ============================================================
# 8. 평가 함수 정의
# ============================================================

# 모델의 손실과 정확도를 계산하는 함수입니다.
def evaluate_model(model, X_tensor, y_tensor):
    # 모델을 평가 모드로 변경합니다.
    # Dropout 같은 계층은 학습 때와 평가 때 동작이 다르므로 반드시 설정해야 합니다.
    model.eval()

    # 평가 시에는 기울기 계산이 필요하지 않으므로 no_grad를 사용합니다.
    # 이렇게 하면 메모리를 절약하고 계산 속도도 빨라집니다.
    with torch.no_grad():
        # 입력 데이터와 정답 데이터를 device로 이동합니다.
        X_tensor = X_tensor.to(device)
        y_tensor = y_tensor.to(device)

        # 모델 예측값 logit을 계산합니다.
        logits = model(X_tensor)

        # 손실값을 계산합니다.
        loss = criterion(logits, y_tensor)

        # logit에 Sigmoid를 적용하여 0~1 사이의 확률로 바꿉니다.
        probabilities = torch.sigmoid(logits)

        # 확률이 0.5 이상이면 1, 아니면 0으로 분류합니다.
        predictions = (probabilities >= 0.5).float()

        # 예측값과 실제값이 같은 비율을 계산합니다.
        accuracy = (predictions == y_tensor).float().mean()

    # loss와 accuracy를 Python 숫자로 변환하여 반환합니다.
    return loss.item(), accuracy.item()

In [ ]:
# ============================================================
# 9. 모델 학습 및 검증
# ============================================================

# 전체 학습 반복 횟수입니다.
epochs = 100

# epoch마다 손실과 정확도를 저장할 리스트입니다.
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# 지정한 epoch 횟수만큼 반복합니다.
for epoch in range(epochs):
    # 모델을 학습 모드로 변경합니다.
    model.train()

    # 한 epoch의 총 손실을 저장할 변수입니다.
    total_train_loss = 0.0

    # DataLoader에서 미니배치 단위로 데이터를 꺼냅니다.
    for batch_X, batch_y in train_loader:
        # 입력 데이터와 정답 데이터를 device로 이동합니다.
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # 이전 미니배치에서 계산된 기울기를 초기화합니다.
        optimizer.zero_grad()

        # 모델에 입력 데이터를 넣어 예측값 logit을 계산합니다.
        logits = model(batch_X)

        # 예측값과 실제값 사이의 손실을 계산합니다.
        loss = criterion(logits, batch_y)

        # 손실값을 기준으로 역전파를 수행하여 기울기를 계산합니다.
        loss.backward()

        # 계산된 기울기를 사용하여 모델 가중치를 업데이트합니다.
        optimizer.step()

        # 현재 미니배치 손실에 샘플 수를 곱해 누적합니다.
        total_train_loss += loss.item() * batch_X.size(0)

    # 한 epoch의 평균 학습 손실을 계산합니다.
    avg_train_loss = total_train_loss / len(train_loader.dataset)

    # 학습 데이터 전체 기준 손실과 정확도를 계산합니다.
    train_loss, train_acc = evaluate_model(model, X_train_tensor, y_train_tensor)

    # 검증 데이터 기준 손실과 정확도를 계산합니다.
    val_loss, val_acc = evaluate_model(model, X_val_tensor, y_val_tensor)

    # 결과를 리스트에 저장합니다.
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    # 10 epoch마다 학습 상태를 출력합니다.
    if (epoch + 1) % 10 == 0:
        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

In [ ]:
# ============================================================
# 10. 학습 곡선 시각화
# ============================================================

# 손실 그래프 크기를 설정합니다.
plt.figure(figsize=(8, 5))

# 학습 손실을 선 그래프로 그립니다.
plt.plot(train_losses, label="Train Loss")

# 검증 손실을 선 그래프로 그립니다.
plt.plot(val_losses, label="Validation Loss")

# 그래프 제목을 설정합니다.
plt.title("Train Loss vs Validation Loss")

# x축 이름을 설정합니다.
plt.xlabel("Epoch")

# y축 이름을 설정합니다.
plt.ylabel("Loss")

# 범례를 표시합니다.
plt.legend()

# 그래프를 출력합니다.
plt.show()

# 정확도 그래프 크기를 설정합니다.
plt.figure(figsize=(8, 5))

# 학습 정확도를 선 그래프로 그립니다.
plt.plot(train_accuracies, label="Train Accuracy")

# 검증 정확도를 선 그래프로 그립니다.
plt.plot(val_accuracies, label="Validation Accuracy")

# 그래프 제목을 설정합니다.
plt.title("Train Accuracy vs Validation Accuracy")

# x축 이름을 설정합니다.
plt.xlabel("Epoch")

# y축 이름을 설정합니다.
plt.ylabel("Accuracy")

# 범례를 표시합니다.
plt.legend()

# 그래프를 출력합니다.
plt.show()

In [ ]:
# ============================================================
# 11. 테스트 데이터 최종 평가
# ============================================================

# 모델을 평가 모드로 변경합니다.
model.eval()

# 테스트 데이터 예측 시 기울기 계산을 하지 않습니다.
with torch.no_grad():
    # 테스트 입력 데이터를 device로 이동합니다.
    X_test_device = X_test_tensor.to(device)

    # 테스트 데이터에 대한 logit을 계산합니다.
    test_logits = model(X_test_device)

    # Sigmoid를 적용해 양성 클래스일 확률로 변환합니다.
    test_probabilities = torch.sigmoid(test_logits).cpu().numpy().ravel()

    # 확률이 0.5 이상이면 1, 아니면 0으로 분류합니다.
    test_predictions = (test_probabilities >= 0.5).astype(int)

# 실제 테스트 정답을 NumPy 배열로 변환합니다.
y_true = y_test

# 정확도를 계산합니다.
accuracy = accuracy_score(y_true, test_predictions)

# 정밀도를 계산합니다.
precision = precision_score(y_true, test_predictions)

# 재현율을 계산합니다.
recall = recall_score(y_true, test_predictions)

# F1-score를 계산합니다.
f1 = f1_score(y_true, test_predictions)

# ROC-AUC를 계산합니다.
# ROC-AUC는 0/1 예측값이 아니라 양성 클래스 확률값을 사용합니다.
auc = roc_auc_score(y_true, test_probabilities)

# 결과를 출력합니다.
print("테스트 정확도 Accuracy:", round(accuracy, 4))
print("테스트 정밀도 Precision:", round(precision, 4))
print("테스트 재현율 Recall:", round(recall, 4))
print("테스트 F1-score:", round(f1, 4))
print("테스트 ROC-AUC:", round(auc, 4))

In [ ]:
# ============================================================
# 12. 혼동행렬 확인
# ============================================================

# 혼동행렬을 계산합니다.
# 행은 실제값, 열은 예측값을 의미합니다.
cm = confusion_matrix(y_true, test_predictions)

# 혼동행렬을 출력합니다.
print("Confusion Matrix")
print(cm)

# 혼동행렬에서 TN, FP, FN, TP를 꺼냅니다.
tn, fp, fn, tp = cm.ravel()

# 각각의 의미를 출력합니다.
print("TN: 실제 0을 0으로 맞춘 개수 =", tn)
print("FP: 실제 0을 1로 잘못 예측한 개수 =", fp)
print("FN: 실제 1을 0으로 잘못 예측한 개수 =", fn)
print("TP: 실제 1을 1로 맞춘 개수 =", tp)

# sklearn의 classification_report로 주요 평가 지표를 한 번에 확인합니다.
print("\nClassification Report")
print(classification_report(y_true, test_predictions, target_names=data.target_names))

In [ ]:
# ============================================================
# 13. ROC Curve 시각화
# ============================================================

# ROC Curve를 그리기 위해 FPR, TPR, Threshold를 계산합니다.
fpr, tpr, thresholds = roc_curve(y_true, test_probabilities)

# 그래프 크기를 설정합니다.
plt.figure(figsize=(8, 5))

# ROC Curve를 그립니다.
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc:.4f})")

# 무작위 예측 기준선을 그립니다.
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Guess")

# 그래프 제목을 설정합니다.
plt.title("ROC Curve")

# x축 이름을 설정합니다.
plt.xlabel("False Positive Rate")

# y축 이름을 설정합니다.
plt.ylabel("True Positive Rate")

# 범례를 표시합니다.
plt.legend()

# 그래프를 출력합니다.
plt.show()

## 정리

이 노트북에서 확인한 핵심 흐름은 다음과 같습니다.

- 학습 데이터는 모델의 가중치를 학습하는 데 사용합니다.
- 검증 데이터는 학습 중 모델이 과적합되는지 확인하는 데 사용합니다.
- 테스트 데이터는 최종 모델 성능을 확인하는 데 사용합니다.
- Accuracy만 보면 불균형 데이터에서 잘못 판단할 수 있으므로 Precision, Recall, F1-score, ROC-AUC도 함께 확인해야 합니다.